In [1]:
import colorsys
import hashlib
import json
import math
import os
from pathlib import Path

import geopandas as gpd
import h3
import h3pandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm
import pydeck as pdk

2025-09-04 08:13:35.185749: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-04 08:13:35.185781: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-04 08:13:35.186931: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-04 08:13:35.192081: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-04 08:13:35.718987: W tensorflow/compiler/tf2

In [2]:
import sys
sys.path.append("../")
from lib.models.geo_model_net import ResLayer
from lib.models.encoders import CoordEncoder

In [3]:
tax_2_23 = pd.read_csv(
    "/data-ssd/alex/datasets/vision-export-20250622060005-aka-2.23/taxonomy.csv"
)

In [4]:
class TFGeoPriorModel:

    def __init__(self, model_path):
        # initialize the geo model for inference
        tf.config.set_visible_devices([], "GPU")
        visible_devices = tf.config.get_visible_devices()
        for device in visible_devices:
            assert device.device_type != "GPU"
        self.gpmodel = tf.keras.models.load_model(
            model_path,
            custom_objects={"ResLayer": ResLayer},
            compile=False
        )

    def predict(self, latitude, longitude):
        encoded_loc = TFGeoPriorModel.encode_loc([latitude], [longitude])
        return self.gpmodel(tf.convert_to_tensor(
            tf.expand_dims(encoded_loc[0], axis=0)
        ), training=False)[0]

    def features_for_one_class(self, latitude, longitude):
        """Evalutes the model for a single class and multiple locations

        Args:
            latitude (list): A list of latitudes
            longitude (list): A list of longitudes (same length as latitude)
            class_of_interest (int): The single class to eval

        Returns:
            numpy array: scores for class of interest at each location
        """

        encoded_loc = TFGeoPriorModel.encode_loc(latitude, longitude)
        loc_emb = self.gpmodel.layers[0](encoded_loc)

        # res layers - feature extraction
        x = self.gpmodel.layers[1](loc_emb)
        x = self.gpmodel.layers[2](x)
        x = self.gpmodel.layers[3](x)
        x = self.gpmodel.layers[4](x)

        return x

    def eval_one_class_from_features(self, features, class_of_interest):
        # process just the one class
        return tf.keras.activations.sigmoid(
            tf.matmul(
                tf.expand_dims(self.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
                features,
                transpose_b=True
            )
        ).numpy()

    @staticmethod
    def encode_loc(latitude, longitude):
        latitude = np.array(latitude)
        longitude = np.array(longitude)
        grid_lon = longitude.astype("float32") / 180.0
        grid_lat = latitude.astype("float32") / 90.0

        norm_loc = tf.stack([grid_lon, grid_lat], axis=1)

        encoded_loc = tf.concat([
            tf.sin(norm_loc * math.pi),
            tf.cos(norm_loc * math.pi),
        ], axis=1)
        return encoded_loc

In [138]:
geo_model_2_23 = TFGeoPriorModel(
    "/home/alex/inatGeoModelTraining/saved_models/saved_model_2_23_coords_elev.h5"
)

In [139]:
california_polygon = {
    "type": "Polygon",
    "coordinates": [[
        [-140.482003, 45.000325],
        [-100.131211, 45.995232],
        [-100.465747, 20.542245],
        [-140.409591, 20.534156],
        [-140.482003, 45.000325]
    ]]
}
resolution = 4
h3_indices = h3.polyfill(california_polygon, resolution, geo_json_conformant=True)
df = pd.DataFrame({
    "h3_04": list(h3_indices)
}).set_index("h3_04")
dfh3 = df.h3.h3_to_geo()
dfh3["lng"] = dfh3.geometry.x
dfh3["lat"] = dfh3.geometry.y
_ = dfh3.pop("geometry")

In [140]:
raster = np.load("/home/alex/sinr/data/env/elev_scaled.npy")
encoder = CoordEncoder(
    encoding_strategy="sinusoidal",
    raster=raster
)
encoded_locs = encoder.encode(dfh3.to_numpy())

In [141]:
class_of_interest = int(tax_2_23[tax_2_23["name"] == "Crotalus ruber"].iloc[0].leaf_class_id)
taxon_of_interest = int(tax_2_23[tax_2_23["name"] == "Crotalus ruber"].iloc[0].taxon_id)

In [142]:
thresholds_2_23 = pd.read_csv(
    "/home/alex/thresholds_2_23_sinr.csv"
)
threshold = thresholds_2_23[thresholds_2_23.taxon_id==taxon_of_interest].iloc[0].thres
threshold

0.13178508

In [143]:
loc_emb = geo_model_2_23.gpmodel.layers[0](encoded_locs)

# res layers - feature extraction
x = geo_model_2_23.gpmodel.layers[1](loc_emb)
x = geo_model_2_23.gpmodel.layers[2](x)
x = geo_model_2_23.gpmodel.layers[3](x)
geo_model_feats = geo_model_2_23.gpmodel.layers[4](x)

raw_preds = tf.matmul(
    tf.expand_dims(geo_model_2_23.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
    geo_model_feats,
    transpose_b=True
)
preds = tf.keras.activations.sigmoid(raw_preds).numpy()

In [144]:
dfh3["preds"] = preds[0]
dfh3.reset_index(inplace=True)
dfh3["fill_color"] = dfh3["preds"].apply(
    lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100]
)
dfh3["preds_thresh"] = dfh3.preds.apply(
    lambda x: 0.5 if x > threshold else 0
)
dfh3["thresh_fill_color"] = dfh3["preds_thresh"].apply(
    lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100]
)

In [145]:
train_data_2_23 = pd.read_parquet(
    "/data-ssd/alex/datasets/vision-export-20250622060005-aka-2.23/spatial_data.parquet"
)

In [146]:
train_data_taxon = train_data_2_23[train_data_2_23.taxon_id==taxon_of_interest]

In [147]:
train_data_taxon.dropna(inplace=True)

train_data_taxon["coordinates"] = train_data_taxon.apply(
    lambda x: [x.longitude, x.latitude],
    axis=1
)

/tmp/ipykernel_927291/4276717046.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data_taxon.dropna(inplace=True)
/tmp/ipykernel_927291/4276717046.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data_taxon["coordinates"] = train_data_taxon.apply(


In [148]:
train_data_taxon.sample(5)

,observation_id,latitude,longitude,observed_on,leaf_class_id,iconic_class_id,spatial_class_id,taxon_id,obs_taxon_id,community,captive,positional_accuracy,coordinates
163782377,39406225,33.6947,-117.6420,2020-02-28,85573.0,5,85573,30724,30724,1,0,9.0,"[-117.642, 33.6947]"
163782522,219376451,32.8235,-117.0538,2024-05-29,85573.0,5,85573,30724,30724,1,0,3.0,"[-117.0538, 32.8235]"
163780080,38671694,24.0748,-109.9927,2020-02-13,85573.0,5,85573,30724,30724,1,0,349.0,"[-109.9927, 24.0748]"
163781314,109306222,33.6744,-117.6441,2022-03-23,85573.0,5,85573,30724,30724,1,0,6.0,"[-117.6441, 33.6744]"
163781819,155992068,32.8179,-117.0647,2023-04-21,85573.0,5,85573,30724,30724,1,0,24.0,"[-117.0647, 32.8179]"


In [149]:
train_data_taxon.dropna(inplace=True)

/tmp/ipykernel_927291/4052754145.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data_taxon.dropna(inplace=True)


In [150]:
# Use pandas to calculate additional data
train_data_taxon["exits_radius"] = 10.0
train_data_taxon.exits_radius = train_data_taxon.exits_radius.astype(float)

/tmp/ipykernel_927291/1622483885.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data_taxon["exits_radius"] = 10.0
/tmp/ipykernel_927291/1622483885.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data_taxon.exits_radius = train_data_taxon.exits_radius.astype(float)


In [151]:
type(train_data_taxon["coordinates"].iloc[0]), type(train_data_taxon.exits_radius.iloc[0])

(list, numpy.float64)

In [152]:
layer_scatter = pdk.Layer(
    "ScatterplotLayer",
    train_data_taxon[["coordinates", "exits_radius"]],
    pickable=True,
    opacity=0.4,
    stroked=True,
    filled=True,
    radius_scale=6,
    radius_min_pixels=1,
    radius_max_pixels=100,
    line_width_min_pixels=1,
    get_position="coordinates",
    get_radius="exits_radius",
    get_fill_color=[255, 140, 0],
    get_line_color=[0, 0, 0],
)

layer_grid = pdk.Layer(
    "H3HexagonLayer",
    dfh3.to_dict("records"),
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="thresh_fill_color",
    line_width_min_pixels=0,
)

In [153]:
# Set the viewport location
view_state_grid = pdk.ViewState(
    latitude=30, 
    longitude=-117, 
    zoom=4, 
    bearing=0, 
    pitch=0
)

In [154]:
r_grid = pdk.Deck(
    layers=[
        layer_grid, 
        layer_scatter
    ], 
    initial_view_state=view_state_grid,
    map_style=pdk.map_styles.LIGHT,
)
#r_grid.show()
r_grid.to_html("crotalus_ruber_r4.html", notebook_display=False)

# ok let's do children at r5

In [155]:
resolution = 5
h3_indices = h3.polyfill(california_polygon, resolution, geo_json_conformant=True)
df = pd.DataFrame({
    "h3_05": list(h3_indices)
}).set_index("h3_05")
dfh3 = df.h3.h3_to_geo()
dfh3["lng"] = dfh3.geometry.x
dfh3["lat"] = dfh3.geometry.y
_ = dfh3.pop("geometry")

In [156]:
locs = dfh3.to_numpy()
encoded_locs = encoder.encode(locs)

In [157]:
loc_emb = geo_model_2_23.gpmodel.layers[0](encoded_locs)

# res layers - feature extraction
x = geo_model_2_23.gpmodel.layers[1](loc_emb)
x = geo_model_2_23.gpmodel.layers[2](x)
x = geo_model_2_23.gpmodel.layers[3](x)
geo_model_feats = geo_model_2_23.gpmodel.layers[4](x)

raw_preds = tf.matmul(
    tf.expand_dims(geo_model_2_23.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
    geo_model_feats,
    transpose_b=True
)
preds = tf.keras.activations.sigmoid(raw_preds).numpy()

In [158]:
dfh3["preds"] = preds[0]
dfh3 = dfh3.h3.h3_to_parent_aggregate(4, operation='mean', return_geometry=False)
dfh3.reset_index(inplace=True)
dfh3["fill_color"] = dfh3["preds"].apply(
    lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100]
)
dfh3["preds_thresh"] = dfh3.preds.apply(
    lambda x: 0.5 if x > threshold else 0
)
dfh3["thresh_fill_color"] = dfh3["preds_thresh"].apply(
    lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100]
)

In [159]:
layer_grid = pdk.Layer(
    "H3HexagonLayer",
    dfh3.to_dict("records"),
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="thresh_fill_color",
    line_width_min_pixels=0,
)
r_grid = pdk.Deck(
    layers=[
        layer_grid, 
        layer_scatter
    ], 
    initial_view_state=view_state_grid,
    map_style=pdk.map_styles.LIGHT,
)
#r_grid.show()
r_grid.to_html("crotalus_ruber_r4_from_r5_agg_avg.html", notebook_display=False)

# ok let's do children at r6

In [160]:
resolution = 6
h3_indices = h3.polyfill(california_polygon, resolution, geo_json_conformant=True)
df = pd.DataFrame({
    "h3_06": list(h3_indices)
}).set_index("h3_06")
dfh3 = df.h3.h3_to_geo()
dfh3["lng"] = dfh3.geometry.x
dfh3["lat"] = dfh3.geometry.y
_ = dfh3.pop("geometry")

In [161]:
locs = dfh3.to_numpy()
encoded_locs = encoder.encode(locs)

In [162]:
loc_emb = geo_model_2_23.gpmodel.layers[0](encoded_locs)

# res layers - feature extraction
x = geo_model_2_23.gpmodel.layers[1](loc_emb)
x = geo_model_2_23.gpmodel.layers[2](x)
x = geo_model_2_23.gpmodel.layers[3](x)
geo_model_feats = geo_model_2_23.gpmodel.layers[4](x)

raw_preds = tf.matmul(
    tf.expand_dims(geo_model_2_23.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
    geo_model_feats,
    transpose_b=True
)
preds = tf.keras.activations.sigmoid(raw_preds).numpy()

In [163]:
dfh3["preds"] = preds[0]
dfh3 = dfh3.h3.h3_to_parent_aggregate(4, operation='mean', return_geometry=False)
dfh3.reset_index(inplace=True)
dfh3["fill_color"] = dfh3["preds"].apply(
    lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100]
)
dfh3["preds_thresh"] = dfh3.preds.apply(
    lambda x: 0.5 if x > threshold else 0
)
dfh3["thresh_fill_color"] = dfh3["preds_thresh"].apply(
    lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100]
)

In [164]:
layer_grid = pdk.Layer(
    "H3HexagonLayer",
    dfh3.to_dict("records"),
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="thresh_fill_color",
    line_width_min_pixels=0,
)
r_grid = pdk.Deck(
    layers=[
        layer_grid, 
        layer_scatter
    ], 
    initial_view_state=view_state_grid,
    map_style=pdk.map_styles.LIGHT,
)
#r_grid.show()
r_grid.to_html("crotalus_ruber_r4_from_r6_agg_avg.html", notebook_display=False)